# 06 — Wire MCP Tools Into the Agent

We have:
- An agent loop with `ReActAgent` and a `ToolRegistry` (notebooks 01-03)
- Two MCP servers: `notes_server` (stdio + HTTP) and `cache_server` (HTTP) (notebooks 04-05)

Now we connect them. The goal is that **the agent loop has no idea** whether a tool is local Python or remote MCP — they all look the same to the loop.

**By the end of this notebook you will:**
1. Build an `MCPToolAdapter` that wraps an MCP tool as a `BaseTool`
2. Register a mix of native tools + MCP tools into one `ToolRegistry`
3. Run a real research mini-agent: search the web → fetch a page → save findings as MCP notes
4. See namespacing (`notes__save_note`) prevent tool-name collisions

Lecture reference: **S7 §3** (lifecycle), **S7 §6.3** (namespacing as defense).


## 1. Read the adapter

`MCPToolAdapter` lives at `src/deepbrief/tools/mcp_adapter.py`. The pattern: subclass `BaseTool`, and in `execute()` open a fresh MCP session, call the tool, parse the response.

In [ ]:
import inspect
from deepbrief.tools.mcp_adapter import MCPToolAdapter, discover_http_tools, _strictify

print(inspect.getsource(MCPToolAdapter))

## 2. Start the MCP servers

Same as notebook 05 — we'll run both `notes_server` (HTTP) and `cache_server` (HTTP) as background subprocesses. Make sure the cells from earlier are stopped first.


In [ ]:
import os, subprocess, sys, time

subprocess.run(["pkill", "-f", "deepbrief.mcp_servers"], capture_output=True)
time.sleep(1)

notes_proc = subprocess.Popen(
    [sys.executable, "-m", "deepbrief.mcp_servers.notes_server", "--http"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    env={**os.environ, "MCP_PORT": "8766"},
)
cache_proc = subprocess.Popen(
    [sys.executable, "-m", "deepbrief.mcp_servers.cache_server"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    env={**os.environ, "PORT": "8765"},
)
time.sleep(2)
for p, name in [(notes_proc, "notes"), (cache_proc, "cache")]:
    print(f"{name}_server: pid={p.pid}  alive={p.poll() is None}")

## 3. Discover MCP tools and register them

`discover_http_tools` connects to the server, lists its tools, returns one `MCPToolAdapter` per tool. We register them in our `ToolRegistry` alongside the native `WebSearchTool` and `FetchURLTool`.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from deepbrief.tools.registry import ToolRegistry
from deepbrief.tools.web_search import WebSearchTool, MockSearchTool
from deepbrief.tools.fetch_url import FetchURLTool

# Use real search if Tavily key set, mock otherwise
search_tool = WebSearchTool() if os.getenv("TAVILY_API_KEY") else MockSearchTool()
print(f"Search backend: {type(search_tool).__name__}")

registry = ToolRegistry()
registry.register(search_tool)
registry.register(FetchURLTool())

# Discover MCP tools and namespace them
for adapter in await discover_http_tools("http://localhost:8766/mcp", "notes"):
    registry.register(adapter)

for adapter in await discover_http_tools("http://localhost:8765/mcp", "cache"):
    registry.register(adapter)

print("\nAll tools in registry:")
for name in registry.list_tools():
    print(f"  • {name}")

Notice the **namespacing**: `notes__save_note` and `cache__cache_get`. The native tools (`web_search`, `fetch_url`) keep simple names because we own them. If we hadn't namespaced, two MCP servers exposing a `search` tool would collide — and we'd silently call the wrong one. Tool-name shadowing is a real attack class (S7 §6.3).


## 4. Verify the adapter works end-to-end

In [ ]:
# Call an MCP tool through our registry, just like a native tool
result = await registry.execute("notes__save_note", {
    "title": "Test from notebook 06",
    "content": "This note was created via the MCPToolAdapter.",
})
print("save_note via registry:", result.output, "  latency_ms:", result.latency_ms)

result = await registry.execute("notes__list_notes", {})
print("\nlist_notes:")
for note in result.output[:3]:   # first 3
    print("  -", note)

## 5. Run the agent with mixed tools

Now the magic: hand the registry to `ReActAgent` and ask it to do a real research task. The agent doesn't know which tools are local Python and which are MCP — they all look the same.

In [ ]:
from deepbrief.agents.react import ReActAgent

SYSTEM_PROMPT = """You are a research assistant building notes for a brief.

# Workflow
1. Use `web_search` to find sources.
2. For the most relevant 1-2 results, call `fetch_url` to read the page.
3. Save concise findings via `notes__save_note`. Include the source URL in the content.
4. When done, list saved notes via `notes__list_notes` and write a brief summary.

# Constraints
- Make at most 3 web_search calls and 2 fetch_url calls.
- Save at most 3 notes total.
- If a tool fails, do not retry more than once.
"""

agent = ReActAgent(
    registry=registry,
    system_prompt=SYSTEM_PROMPT,
    max_steps=10,
)
result = await agent.run("What is the Model Context Protocol (MCP) and who created it?")

print("=== ANSWER ===")
print(result.answer)
print(f"\nsteps={result.steps}  terminated_by={result.terminated_by}")
print("\n=== TRACE ===")
for entry in result.trace:
    calls = entry['tool_calls'] or '—'
    print(f"  step {entry['step']}: {calls}")

Look at the trace — you should see a mix of native tool calls (`web_search`, `fetch_url`) and MCP tool calls (`notes__save_note`, `notes__list_notes`). The agent treats them identically.


## 6. Verify the notes ended up on disk

Notes are persisted as JSON files in `./data/notes/` (gitignored). The cache lives in-memory in the cache_server process — restarting the cache_server clears it.

In [ ]:
from pathlib import Path
import json

notes_dir = Path("./data/notes")
if notes_dir.exists():
    for path in sorted(notes_dir.glob("*.json"))[-5:]:
        data = json.loads(path.read_text())
        print(f"📝 {data['title']}  ({data['id']})")
        print(f"   {data['content'][:150]}...\n")

## 7. Cleanup

In [ ]:
for proc, name in [(notes_proc, "notes"), (cache_proc, "cache")]:
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()
    print(f"{name}_server stopped")

## 8. Self-check

1. Why does `MCPToolAdapter.execute()` open a *fresh* session per call instead of reusing one?
2. What's the namespacing convention and what attack does it defend against?
3. The agent calls `notes__save_note`. Walk through what happens on the wire (which transport, which JSON-RPC methods).
4. Why do we patch `additionalProperties: False` onto MCP-discovered schemas (`_strictify`)?

## What's next

Notebook **07** — we go horizontal. Instead of one big agent, build a **coordinator + specialist** team that talks via the **A2A protocol**. Each agent has its own LLM, its own tool stack, and its own little loop. Coordinator uses A2A to delegate; each specialist uses MCP to access tools.